In [4]:
# 2. 필요한 모듈 임포트
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

from services.data_preparer import prepare_proposal_17_data
from services.proposal_views.proposal_17_view import create_figure_and_df

# --- 테스트 실행 ---

# 1. app.py의 역할 (1): 데이터 준비 함수를 호출하여 필요한 데이터를 미리 로드합니다.
print("Step 1: `data_preparer`를 통해 분석용 데이터와 순서 정보를 준비합니다...")
# 글로벌 필터는 모두 기본값('전체')으로 호출
data_bundle = prepare_proposal_17_data() 
order_map = data_bundle.get("order_map", {})
print(" -> 데이터 준비 완료!")


# 2. app.py의 역할 (2): 사용자가 Streamlit 위젯을 통해 필터를 선택했다고 가정합니다.

# 2-1. app.py에 있을 차원 설정(Dimension Config) 정의
DIMENSION_CONFIG = {
    '부서별': {'type': 'hierarchical', 'top': 'DIVISION_NAME', 'sub': 'OFFICE_NAME'},
    '직무별': {'type': 'hierarchical', 'top': 'JOB_L1_NAME', 'sub': 'JOB_L2_NAME'},
    '직위직급별': {'type': 'hierarchical', 'top': 'POSITION_NAME', 'sub': 'GRADE_ID'},
    '성별': {'type': 'flat', 'col': 'GENDER'},
    '연령별': {'type': 'flat', 'col': 'AGE_BIN'},
    '경력연차별': {'type': 'flat', 'col': 'CAREER_BIN'},
    '연봉구간별': {'type': 'flat', 'col': 'SALARY_BIN'},
    '지역별': {'type': 'flat', 'col': 'REGION_CATEGORY'},
    '계약별': {'type': 'flat', 'col': 'CONT_CATEGORY'}
}

# 2-2. 사용자 선택 시뮬레이션
# ----- 테스트하고 싶은 값으로 변경 -----
selected_dimension_ui = '경력연차별'
drilldown_selection = '전체'
# -----------------------------------

print(f"Step 2: 사용자가 '{selected_dimension_ui}' 차원을, '{drilldown_selection}' 그룹으로 보기를 선택했습니다.")


# 3. app.py의 역할 (3): view 함수에 준비된 모든 데이터와 필터 값을 전달하여 결과물 생성
print("Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...")
# 이 뷰는 data_bundle 전체를 필요로 하므로, analysis_df 대신 data_bundle을 전달
if data_bundle:
    fig, aggregate_df = create_figure_and_df(
        data_bundle=data_bundle, 
        dimension_ui_name=selected_dimension_ui, 
        drilldown_selection=drilldown_selection,
        dimension_config=DIMENSION_CONFIG,
        order_map=order_map
    )
    print(" -> 생성 완료!")
else:
    print(" -> 분석할 데이터가 없어 빈 결과물을 생성합니다.")
    fig, aggregate_df = go.Figure(), pd.DataFrame()


# --- 결과 확인 ---

# 4. ipynb에서 생성된 그래프를 확인합니다.
print("\n--- [결과 1] 생성된 Plotly 그래프 ---")
# pio.renderers.default = 'vscode' 
fig.show()

# 5. ipynb에서 생성된 요약 테이블(aggregate_df)을 확인합니다.
print(f"\n--- [결과 2] 생성된 요약 테이블 ---")
display(aggregate_df)

Step 1: `data_preparer`를 통해 분석용 데이터와 순서 정보를 준비합니다...
 -> 데이터 준비 완료!
Step 2: 사용자가 '경력연차별' 차원을, '전체' 그룹으로 보기를 선택했습니다.
Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...
 -> 생성 완료!

--- [결과 1] 생성된 Plotly 그래프 ---


/app/src/services/proposal_views/proposal_17_view.py:125: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

/app/src/services/proposal_views/proposal_17_view.py:125: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['17.3' '16.9' '17.3' '17.3' '17.0']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.

/app/src/services/proposal_views/proposal_17_view.py:125: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['18.0' '17.7' '16.2' '20.6' '17.8']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.

/app/src/services/proposal_views/proposal_17_view.py:125: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['13.0' '11.7' '11.8' '12.5' '13.0']' 


--- [결과 2] 생성된 요약 테이블 ---


CAREER_BIN                 전체   1년 미만   1~3년   3~7년  7~15년 15년 이상
평균 초과근무시간(분) Monday      17.3    18.0   13.0   17.0   17.9   20.2
             Tuesday     16.9    17.7   11.7   16.6   17.6   19.3
             Wednesday   17.3    16.2   11.8   16.6   18.1   21.6
             Thursday    17.3    20.6   12.5   16.4   18.1   20.8
             Friday      17.0    17.8   13.0   16.0   18.0   19.5
요일별 연차사용률(%) Monday     6.51%  20.25%  8.38%  6.41%  6.16%  6.24%
             Tuesday    6.43%  18.87%  7.29%  6.48%  6.15%  6.08%
             Wednesday  6.59%  18.02%  8.07%  6.52%  6.32%  6.15%
             Thursday   6.45%  18.29%  8.39%  6.37%  6.11%  5.94%
             Friday     6.59%  20.68%  8.02%  6.71%  6.14%  6.27%